# GLLM fine-tuning using Low-Rank Adaption (LoRA)

This notebook has the purpose of training a local GLLM for the MAP classification task (in our case Qwen-4B-Instruct) using LoRA. LoRA is a technique that reduces the number of trainable parameters during fine-tuning. It freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture (see Hu et al. (2021) - https://arxiv.org/abs/2106.09685). Broadly summarized, it reduces the number of trainable parameter by approx. 10,000 tunes and, hence, the required GPU memory by a factor of 3. 

<div class="alert-warning">
Libraries
</div>

In [ ]:
# Run the following commands in your terminal to install LlamaFactory and its dependencies (for more information, visit the github repository):
# Remember where you cloned the repository and change the path below accordingly (variable "llama_factory_root").
# git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git 
# cd LlamaFactory
# pip install -e .
# pip install -r requirements/metrics.txt

First, import the necessary python packages.

In [ ]:
#Import the necessary packages
import os
import re
import copy
import json
from pathlib import Path
import pandas as pd
import getpass
import datasets
import tempfile
import subprocess
import plotnine
import time
import llamafactory as lf
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import optuna
import scipy.stats as stats
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

<div class="alert-warning">
Set the working directory (!IMPORTANT: YOU NEED TO CHANGE THE ROOT FOR THE LLAMAFACTORY DIRECTORY TO THE ACTUAL!)
</div>

Next, we set the working directories on the HPC, define the cache directory for the HuggingFace models, and login to HuggingFace to be able to download the inference models.

In [ ]:
# Set working directory 
os.chdir('../../../../data')

# Define the directory where HuggingFace can save/load the models to
transformers_cache_dir = "../hf_cache"

# Create the directory if it does not exist
if not os.path.exists(transformers_cache_dir):
    os.makedirs(transformers_cache_dir)

# set the root of the cloned LlamaFactory repository
###############################################################################################################
#!IMPORTANT: Change <path_to_llama_factory> below to the location where you cloned the LlamaFactory repository#
###############################################################################################################
#llama_factory_root = <path_to_llama_factory> #HERE, Example: "C:/Users/<username>/LLaMA-Factory"

### Login to HuggingFace to be able to download local models 
if 'HUGGINGFACE_ACCESS_TOKEN' not in os.environ:
    os.environ['HUGGINGFACE_ACCESS_TOKEN'] = getpass.getpass(prompt='Enter your HuggingFace API key: ')
    
huggingface_access_token = os.environ['HUGGINGFACE_ACCESS_TOKEN']

login(token = huggingface_access_token)

<div class="alert-warning">
Check wether GPU is available for GLLM fine-tuning (!IMPORTANT: YOU NEED TO CHANGE THE PATH TO YOUR ACTUAL CUDA INSTALLATION, IF NECESSARY!)
</div>

Lastly, we will check whether the computing node we are connected to also recognizes the GPU(s) we selected.

In [ ]:
# Check if CUDA is available and print the GPU information
print("PyTorch CUDA available:", torch.cuda.is_available())
print("CUDA version (from torch):", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("Cuda device count:", torch.cuda.device_count())

# Set the random seed to ensure reproducible results across different runs.
torch.random.manual_seed(0)

# Set MKL threading layer to GNU to avoid potential conflicts
os.environ["MKL_THREADING_LAYER"] = "GNU"

# In addition, we need to make sure that the CUDA_HOME environment variable is set correctly. You can do this by running the following command:

print("CUDA_HOME:", os.environ.get("CUDA_HOME"))

# If the output is None, you can set the CUDA_HOME environment variable by running the following lines 
###################################################################################
#!IMPORTANT: Replace <path_to_cuda> with the actual path to your CUDA installation#
###################################################################################

if os.environ.get("CUDA_HOME") is None:
    # Set the CUDA_HOME environment variable
    os.environ["CUDA_HOME"] = <path_to_cuda>  #HERE
    os.environ["PATH"] = os.environ["CUDA_HOME"] + "/bin:" + os.environ["PATH"]
    os.environ["LD_LIBRARY_PATH"] = os.environ["CUDA_HOME"] + "/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
    
    print("CUDA_HOME set to:", os.environ["CUDA_HOME"])

## Prepare training and validation dataset

Before we can run the fine-tuning, we need to adjust our evaluation dataset such that it can be used for model training. (This part can be skipped. For replication use the resulting evaluation set at the end of this section ("evaluation_set_MAP_sentences_final_v2_2"). This part is the same as in the notebook "OpenAI_Fine_Tuning_final.ipynb")

First, we need to add the confidence score to our evaluation dataset. To do so, we load the evaluation dataset and the outputs of our three best-performing zero shot models. Then we test whether the confidence scores of the three models are broadly in line with our measure of "disclosure quality". If higher disclosure quality is associated with larger confidence scores, then we can use them for training. (For human annotator it is easier to say whether the informational quality of a sentence is "low", "medium", or "high" than coming up with a condfidence score)

In [ ]:
# 0. We first load the evaluation set v4 and the outcomes from the best performing models (GPT-4.1, Llama 70B, Qwen 230B) to create a confidence score for each sentence in the evaluation set
# NOTE: The output results of the best performing models (GPT-4.1, Llama 70B, Qwen 230B) are generated using the final inference prompts (which are slightly different from the prompt engeneering prompts)
# The respective code for generating these outputs can be made available upon request. 

df = pd.read_excel('GLLM/evaluation_set_MAP_sentences_final.xlsx')
df_gpt4 = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_gpt-4.1-2025-04-14.xlsx')
df_llama = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_Llama-3.3-70B-Instruct.xlsx')
df_qwen = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_Qwen3-235B-A22B-Instruct-2507-FP8.xlsx')

# 1. We rename the first column in each dataset to "custom_id" and merge the model outputs 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score' with the evaluation set

df_gpt4 = df_gpt4.rename(columns={df_gpt4.columns[0]: 'custom_id'})
df_llama = df_llama.rename(columns={df_llama.columns[0]: 'custom_id'})
df_qwen = df_qwen.rename(columns={df_qwen.columns[0]: 'custom_id'}) 

# For the human annotated dataset we create the new column 'custom_id' by taking the row index as the value for each row, and place it as the first column in the dataframe
df.insert(0, 'custom_id', df.index)
df['LLM_Confidence_Score'] = float(0) 

df_merged = df.merge(df_gpt4[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_gpt4'))
df_merged = df_merged.merge(df_llama[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_llama'))
df_merged = df_merged.merge(df_qwen[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_qwen'))

del df_gpt4, df_llama, df_qwen, df

# 2. We create a new dataframe to calculate the mean confidence score for each model and disclosure quality level of the human annotator
# The rows are the different disclosure quality levels "low", "medium", "high"
# The columns are the different models "gpt4", "llama", "qwen
columns = ['Mean_Confidence_Score_gpt4', 'Mean_Confidence_Score_llama', 'Mean_Confidence_Score_qwen']
rows = ['Low', 'Medium', 'High']
confidence_scores = pd.DataFrame(float(0), index=rows, columns=columns)

for level in rows:
    for model in ['gpt4', 'llama', 'qwen']:
        subset = df_merged[df_merged['Disclosure_quality'] == level]
        mean_confidence = subset[f'LLM_Confidence_Score_{model}'].mean()
        confidence_scores.at[level, f'Mean_Confidence_Score_{model}'] = mean_confidence

print(confidence_scores)

# 3. Next we test whether the confidence scores differ significantly between the different disclosure quality levels for each model using ANOVA

low_scores = df_merged[df_merged['Disclosure_quality'] == 'Low'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]
medium_scores = df_merged[df_merged['Disclosure_quality'] == 'Medium'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]
high_scores = df_merged[df_merged['Disclosure_quality'] == 'High'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]

for model in ['gpt4', 'llama', 'qwen']:
    # Perform ANOVA test low vs. medium
    f_statistic, p_value = stats.f_oneway(low_scores[f'LLM_Confidence_Score_{model}'], medium_scores[f'LLM_Confidence_Score_{model}'], high_scores[f'LLM_Confidence_Score_{model}'])
    print(f'ANOVA results "low" vs "medium" for {model}: F-statistic = {f_statistic}, p-value = {p_value}')
    #Perform ANOVA test medium vs. high
    f_statistic, p_value = stats.f_oneway(medium_scores[f'LLM_Confidence_Score_{model}'], high_scores[f'LLM_Confidence_Score_{model}'])
    print(f'ANOVA results "medium" vs "high" for {model}: F-statistic = {f_statistic}, p-value = {p_value}')

Next, we compute the confidence scores for model fine-tuning. (Note: It yields evaluation set v5, which is based on v4 but also includes the Confidence Score. The score is based on the output of GPT 4.1, Llama 70B, and Qwen 230B where the score is the average from at least two models that have the same output for explicit and implicit as the human annotator and manually adapted where no model has the same output)

In [ ]:
# 1. Create a new column for each model indicating whether the model's output matches the human annotation
df_merged['gpt4_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_gpt4']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_gpt4'])).astype(int)
df_merged['llama_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_llama']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_llama'])).astype(int)
df_merged['qwen_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_qwen']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_qwen'])).astype(int)

# 2. Create a column that indicates the the number of models that do match the human annotation
df_merged['num_match'] = df_merged['gpt4_match'] + df_merged['llama_match'] + df_merged['qwen_match']


# 3. Calculate the average confidence score of the models confidence score for each sentence where at least two model's output matches the human annotation. 
# The output is stored in the column 'LLM_Confidence_Score' and should be in steps of 5 (e.g. 0, 5, 10, ..., 100). If less than two models match the human annotation, the confidence score is set to 0.
def calculate_average_confidence(row):
    confidence_scores = []
    if row['gpt4_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_gpt4'])
    if row['llama_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_llama'])
    if row['qwen_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_qwen'])
    
    if len(confidence_scores) >= 2:
        average_score = np.mean(confidence_scores)
        # Round to nearest multiple of 5
        rounded_score = round(average_score / 5) * 5
        return int(rounded_score)
    else:
        return int(0)

df_merged['LLM_Confidence_Score'] = df_merged.apply(calculate_average_confidence, axis=1)

# 4. Create a column that indicates whether manual adjustment is necessary (num_match < 2, or confidence score < 50 but Implicit 'Yes' or < 70 and Explicit 'Yes')
def needs_manual_adjustment(row):
    if row['num_match'] < 2:
        return 'Yes'
    if row['LLM_Confidence_Score'] < 50 and row['Implicit_MAP_referral'] == 'Yes':
        return 'Yes'
    if row['LLM_Confidence_Score'] < 70 and row['Explicit_MAP_referral'] == 'Yes':
        return 'Yes'
    if row['Explicit_MAP_referral'] == 'No' and row['Implicit_MAP_referral'] == 'No' and row['LLM_Confidence_Score'] >= 50:
        return 'Yes'
    return 'No'

df_merged['manual_adjustment'] = df_merged.apply(needs_manual_adjustment, axis=1)

# 5. Save the updated evaluation set with confidence scores and manual adjustment flags
df_merged.to_excel('GLLM/Fine_tuning_data/evaluation_set_MAP_sentences_final_v2.xlsx', index=False)


## Import training and validation set

First, we need to load the evaluation dataset and split it into a training and validation set for the fine-tuning task. We set the random state of the sample function to the same root (42) as for the OpenAI fine-tuning to receive the same samples.

In [ ]:
# 1. Load the evaluation set v2_2 which includes the confidence scores (with manual adjustments)
df = pd.read_excel('GLLM/Fine_tuning_data/evaluation_set_MAP_sentences_final_v2_2.xlsx')

# 2. Create a new column merging the MAP dimensions from columns "MAP_dimension_1", "MAP_dimension_2", "MAP_dimension_3", and "MAP_dimension_4" into one string separated by commas
def merge_map_dimensions(row):
    return ", ".join([row[f"MAP_dimension_{i}"].strip() for i in range(1, 5) if pd.notna(row[f"MAP_dimension_{i}"])])
df['MAP_dimensions_merged'] = df.apply(merge_map_dimensions, axis=1)

# 3. Save the merged dimensions column as new dataset 
df.to_excel('GLLM/Fine_tuning_data/training_and_validation_data_MAP_sentences.xlsx', index=False)

# 4. Split the data into training and validation set and save as separate dataframes (80% train, 20% validation)
# Set random seed for reproducibility
train_df = df.sample(frac=0.8, random_state=42)

train_df.to_excel('GLLM/Fine_tuning_data/training_data_MAP_sentences.xlsx', index=False)

validation_df = df.drop(train_df.index)

validation_df.to_excel('GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx', index=False)

# 3. create a jsonl file for training and validation set in the required format for LlamaFactory fine-tuning
training_file_name = "GLLM/Fine_tuning_data/training_data_MAP_sentences_Local.jsonl"

training_file = open(training_file_name, "w", encoding="utf-8")

# Iterate through each row in the training dataframe and write the entries (prompt and completion) to the jsonl file
for index, row in train_df.iterrows():
    merged_dimension = row['MAP_dimensions_merged']
    entry = {
        "prompt": f"Text:\n{row['Sentence']}",
        "completion": f"{{\n  \"Explicit_MAP_referral\": \"{row['Explicit_MAP_referral']}\",\n  \"Implicit_MAP_referral\": \"{row['Implicit_MAP_referral']}\",\n  \"Dimension\": \"{merged_dimension}\",\n  \"Confidence_Score\": {row['LLM_Confidence_Score']}\n}}"
    }
    training_file.write(json.dumps(entry) + "\n")
    training_file.flush()

training_file.close()

# Do the same for the validation set
validation_file_name = "GLLM/Fine_tuning_data/validation_data_MAP_sentences_Local.jsonl"
validation_file = open(validation_file_name, "w", encoding="utf-8")
for index, row in validation_df.iterrows():
    merged_dimension = row['MAP_dimensions_merged']
    entry = {
        "prompt": f"Text:\n{row['Sentence']}",
        "completion": f"{{\n  \"Explicit_MAP_referral\": \"{row['Explicit_MAP_referral']}\",\n  \"Implicit_MAP_referral\": \"{row['Implicit_MAP_referral']}\",\n  \"Dimension\": \"{merged_dimension}\",\n  \"Confidence_Score\": {row['LLM_Confidence_Score']}\n}}"
    }
    validation_file.write(json.dumps(entry) + "\n")
    validation_file.flush()

validation_file.close()

Afterwards, we need to add the dataset to the LlaMA-Factory root, i.e., the internal dataset info .json file. This step is important, since the LlaMa-Factory trainer needs to know where the training and validation data are stored.

In [ ]:
# Set path to "dataset_info.json", which is located in the "data" folder of the cloned LlamaFactory repository
# This file is used to specify the training and validation datasets for fine-tuning in LlamaFactory
data_info_path = Path(llama_factory_root) / "data" / "dataset_info.json"

# Get full path of the training and validation jsonl files
full_path_training = os.path.abspath(training_file_name)
full_path_validation = os.path.abspath(validation_file_name)

# Update the dataset_info.json file with the paths to the training and validation datasets
with open(data_info_path, "r") as f:
    dataset_info = json.loads(f.read())

dataset_info["my_data"] = {
    "file_name" : full_path_training,
    "columns" : {
        "prompt" : "prompt",
        "response" : "completion"
    }
}

dataset_info["eval_data"] = {
    "file_name" : full_path_validation,
    "columns" : {
        "prompt" : "prompt",
        "response" : "completion"
    }
}

with open(data_info_path, "w") as f:
    json.dump(dataset_info, f)

## Start fine-tuning the Qwen-4B-Instruct model using LoRA

First, we define the base configuration of the training parameters. Most of these parameters are default values or based on De Kok (2025) - see GitHub page (https://github.com/TiesdeKok/chatgpt_paper/blob/main/code_examples/runpod/finetune.ipynb) and a paper on instruction tuning by Wang et al. (2023) (https://arxiv.org/abs/2310.04793)

In [ ]:
# Create output directory for the fine-tuned model, if it does not exist
optuna_output_dir = "GLLM/Fine_tuning_data/optuna_runs"
if not os.path.exists(optuna_output_dir):
    os.makedirs(optuna_output_dir)

base_config = {
    "stage": "sft",
    "do_train": True, 
    "do_eval": True, # Enable evaluation during training, which is useful for hyperparameter tuning
    "plot_loss": True, # Plot training and evaluation loss curves
    "packing": False, 
    "enable_thinking": False,
    "include_num_input_tokens_seen": True,
    "model_name_or_path": "Qwen/Qwen3-4B-Instruct-2507",
    "dataset_dir": f"{llama_factory_root}/data",
    "dataset": "my_data",
    "eval_dataset": "eval_data",
    "output_dir": optuna_output_dir,
    "preprocessing_num_workers": 16, # Number of workers for data preprocessing
    "finetuning_type": "lora", # Use LoRA "lora" model fine-tuning
    "optim": "adamw_torch", # Optimizer type (default: "adamw_torch")
    "template": "qwen3_nothink", # Template name for data formatting
    "flash_attn": "auto", # Use Flash Attention if available
    "cutoff_len": 2048, # Maximum sequence length
    "learning_rate": 1e-4, # Learning rate for fine-tuning (typically between 1e-4 and 1e-5 for full fine-tuning)
    "num_train_epochs": 5, # Number of training epochs (For a small dataset size of 1800 entries, 3-5 epochs is usually sufficient)
    "per_device_train_batch_size": 8, # Batch size per device during training (depends on GPU memory, typically between 4 and 16)
    "gradient_accumulation_steps": 4, # Number of gradient accumulation steps (typically between 1 and 8)
    "lora_rank": 8, # LoRA rank (typically between 4 and 16)
    "lora_alpha": 32, # LoRA alpha (typically between 8 and 32) - rule of thumb: lora_alpha should be 2-4 times lora_rank
    "max_grad_norm": 1.0, # Maximum gradient norm for clipping (usually set to 1.0)
    "lora_dropout": 0, # LoRA dropout rate (typically between 0 and 0.1)
    "lr_scheduler_type": "linear", # Learning rate scheduler type ("linear", "cosine", "cosine_with_restarts", etc.)
    "logging_steps": 10, # Frequency of logging training metrics
    "fp16": False, # Use mixed precision training if bf16 is not supported (fp16 is less precise than bf16, but more widely supported and memory efficient)
    "bf16": True, # Use bfloat16 precision if supported by the GPU (since we are using a H200, we can use bf16)
    "trust_remote_code": True, 
    "warmup_ratio": 0, # Warmup ratio for learning rate scheduling (typically between 0 and 0.2)
    "lora_target": ["q_proj", "v_proj"], # Target modules for LoRA fine-tuning ("all" or ["q_proj", "v_proj"])
    "eval_strategy": "steps", # Evaluation strategy during training
    "eval_steps": 10, # Evaluate the model every 10 training steps
    "per_device_eval_batch_size": 64 # Batch size per device during evaluation
}

with open("GLLM/Fine_tuning_data/ft_base_config.json", "w") as f:
    json.dump(base_config, f)

tmp_config_file_path = "GLLM/Fine_tuning_data/ft_base_config.json"

Next, we define some helper functions for pipeline inference, json extraction, and performance evaluation.

In [ ]:
# Function to run batched pipeline inference
def run_batched_pipeline_interference(batch, batch_size, generate_text_pipeline, output_column):

    sentences = batch['Sentence']

    messages_batch = [
        [
            {"role": "user", "content": f"Text:\n{sentence}"}
        ]
        for sentence in sentences
    ]

    responses = generate_text_pipeline(messages_batch, batch_size=batch_size)

    #free gpu memory
    torch.cuda.empty_cache()

    return {output_column: [response[0]['generated_text'][-1]['content'].strip() for response in responses]}
# Function to extract JSON fields from LLM output
def extract_json(row):
    """
    Extracts MAP classification fields from LLM output.
    If parsing fails, returns None for all and prints the problematic LLM output.
    """
    llm_output = row["llm_full_output"]
    try:
        json_match = re.search(r'\{.*?\}', llm_output, re.DOTALL)

        if not json_match:
            print("Failed to extract JSON from LLM output:\n", llm_output)
            return {          
                "LLM_Explicit_MAP_referral": None,
                "LLM_Implicit_MAP_referral": None,
                "LLM_Dimension": None,
                "LLM_Confidence_Score": None
            }

        # Parse JSON
        json_str = json_match.group(0)
        result = json.loads(json_str)

        # Ensure the probability is an integer between 0 and 100
        probability = int(result["Confidence_Score"])
        if not (0 <= probability <= 100):
            raise ValueError("Confidence_Score is out of range")

        # Normalize values
        def normalize(value):
            if isinstance(value, str):
                value = value.strip()
                if value.lower() in ["n/a", "", "na", "none", "nan"]:
                    return None
            return value

        return {
            "LLM_Explicit_MAP_referral": normalize(result.get("Explicit_MAP_referral")).capitalize() if normalize(result.get("Explicit_MAP_referral")) else None,
            "LLM_Implicit_MAP_referral": normalize(result.get("Implicit_MAP_referral")).capitalize() if normalize(result.get("Implicit_MAP_referral")) else None,
            "LLM_Dimension": normalize(result.get("Dimension")),
            "LLM_Confidence_Score": probability
        }

    except Exception as e:
        print(f"Error parsing JSON: {e}\nLLM Output:\n{llm_output}\n")
        return {
            "LLM_Explicit_MAP_referral": None,
            "LLM_Implicit_MAP_referral": None,
            "LLM_Dimension": None,
            "LLM_Confidence_Score": None
        }
# Function to evaluate LLM predictions
def evaluate_llm(dataset, model_id, system_idx=0, user_idx=3,
                          truth_exp_col="Explicit_MAP_referral", pred_exp_col="LLM_Explicit_MAP_referral",
                          truth_imp_col="Implicit_MAP_referral", pred_imp_col="LLM_Implicit_MAP_referral",
                          truth_dimension_col="MAP_dimension_1", pred_dimension_col="LLM_Dimension"):

    # Create a copy of the dataset to avoid modifying the original
    df = dataset.copy()
    
    # Drop rows with missing LLM predictions
    df_exp = df.dropna(subset=[truth_exp_col, pred_exp_col]).copy()
    df_imp = df.dropna(subset=[truth_imp_col, pred_imp_col]).copy()

    print(f"Dropped {len(df)-len(df_exp)} explicit / {len(df)-len(df_imp)} implicit sentences")
    
    #Evaluate Explicit MAP Referral
    print("=== Explicit MAP Referral Evaluation ===")
    if not df_exp.empty:
        exp_accuracy = accuracy_score(df_exp[truth_exp_col], df_exp[pred_exp_col])
        exp_f1_yes = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_precision_yes = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_recall_yes = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_f1_no = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_precision_no = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_recall_no = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Explicit):")
        print(classification_report(df_exp[truth_exp_col], df_exp[pred_exp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Explicit MAP evaluation.")
    
    #Evaluate Implicit MAP Referral
    print("\n=== Implicit MAP Referral Evaluation ===")
    if not df_imp.empty:
        imp_accuracy = accuracy_score(df_imp[truth_imp_col], df_imp[pred_imp_col])
        imp_precision_yes = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_recall_yes = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_f1_yes = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_precision_no = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_recall_no = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_f1_no = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Implicit):")
        print(classification_report(df_imp[truth_imp_col], df_imp[pred_imp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Implicit MAP evaluation.")

    #Evaluate MAP Dimension
    print("\n=== MAP Dimension Evaluation ===")
    filtered_dimension = df[
        (df["LLM_Explicit_MAP_referral"] == "No") &
        (df["LLM_Implicit_MAP_referral"] == "No") &
        (~df["LLM_Dimension"].isna())
    ]
    dim_percentage = len(filtered_dimension)/len(df)*100
    print(f"Number of rows where LLM says 'No' to both Explicit and Implicit MAP referral but MAP Dimension is not None: {dim_percentage:.0f}%")

    # Create a list of all dimension columns
    dimension_cols = [col for col in df.columns if col.startswith("MAP_dimension")]
    
    # Check if the micro / macro F1 score for the dimension column 
    label_space = ["Budgeting / Planning", "Cost", "Financing / Investment", "Operations", "Performance / Internal Reporting", "Risk / Internal Control", "Strategy", "Pricing & Revenue Management"]
    
    def encode_labels(text, label_space):
        labels = [l.strip() for l in text.split(",")]
        return [1 if label in labels else 0 for label in label_space]
    
    # join the true cols without empty cells to one column and encode the true and pred dimension cols

    df["dimension_true"] = df[dimension_cols].apply(lambda row: ", ".join(row.dropna().astype(str)), axis=1)
    df["dimension_true_encoded"] = df["dimension_true"].apply(lambda text: encode_labels(text, label_space))
    df["dimension_pred_encoded"] = df[pred_dimension_col].apply(lambda text: encode_labels(text, label_space) if isinstance(text, str) else [0]*len(label_space))

    dimension_micro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="micro", zero_division=0)
    dimension_macro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="macro", zero_division=0)
    dimension_accuracy = accuracy_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist())
    #full report as table
    dimension_full_report = classification_report(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), target_names=label_space, zero_division=0, digits=3)
    
    # In addition, we check if at least one true dimension is included in the predicted dimensions (which may be a comma-separated string)
    def check_dimension_match(row):
        #true_dim = row[truth_dimension_col]
        true_dims = [row[col] for col in dimension_cols if pd.notnull(row[col])]
        pred_raw = row[pred_dimension_col]
        if all(pd.isnull(true_dims)) and pd.isnull(pred_raw):
            return True  # Both are NaN = match
        elif pd.isnull(pred_raw):
            return False  # No prediction = no match
        else:
            pred_dims = [dim.strip() for dim in pred_raw.split(",")]
            return any(true_dim in pred_dims for true_dim in true_dims)

    if not df.empty:
        print("\nFull per-label Dimension classification report:\n")
        print(dimension_full_report)
        print(f"MAP Dimension Accuracy:{dimension_accuracy:.3f}")
        df["dimension_match"] = df.apply(check_dimension_match, axis=1)
        dimension_accuracy_alternative = df["dimension_match"].mean()
        print(f"MAP Dimension Accuracy (both N/A = match, at least one match): {dimension_accuracy_alternative:.3f}")
    else:
        print("No valid rows for MAP Dimension evaluation.")

    # Return the evaluation results as a dictionary
    return {
        "model_id": model_id,
        "system_idx": system_idx,
        "user_idx": user_idx,
        "dropped_explicit": len(df) - len(df_exp),
        "dropped_implicit": len(df) - len(df_imp),
        "explicit_accuracy": exp_accuracy,
        "explicit_precision_yes": exp_precision_yes,
        "explicit_recall_yes": exp_recall_yes,
        "explicit_f1_yes": exp_f1_yes,
        "explicit_precision_no": exp_precision_no,
        "explicit_recall_no": exp_recall_no,
        "explicit_f1_no": exp_f1_no,
        "implicit_accuracy": imp_accuracy,
        "implicit_precision_yes": imp_precision_yes,
        "implicit_recall_yes": imp_recall_yes,
        "implicit_f1_yes": imp_f1_yes,
        "implicit_precision_no": imp_precision_no,
        "implicit_recall_no": imp_recall_no,
        "implicit_f1_no": imp_f1_no,
        "dimension_percentage_false": dim_percentage,
        "dimension_micro_f1": dimension_micro_f1,
        "dimension_macro_f1": dimension_macro_f1,
        "dimension_accuracy": dimension_accuracy,
        "dimension_accuracy_alternative": dimension_accuracy_alternative,
        "dimension_full_report": dimension_full_report
    }

Then, we will wrap a LlamaFactory run in an Optuna trail to test different hyperparameters. In a first step, we define the objective function for Optuna (it should maximize the dimension micro f1 of the validation dataset). The second step runs the hyperparameter fine-tuning.

In [ ]:
# Define the objective function for Optuna
def objective(trial, validation_data):
    # check if output directory for this trial exists, if not create it
    trial_output_dir = f"./{optuna_output_dir}/trial_{trial.number}"
    if not os.path.exists(trial_output_dir):
        os.makedirs(trial_output_dir)

    # Sample hyperparameters
    lr = 1e-4
    rank = 8 # dafault value
    warmup_rate = trial.suggest_categorical("warmup_ratio", [0, 0.03, 0.1, 0.2]) # typically between 0 and 0.2 - tested: 0, 0.1, 0.2
    alpha = trial.suggest_categorical("lora_alpha", [16, 32]) # typically set to 2*lora_rank or 4*lora_rank (tested: 16 & 32)
    epochs = trial.suggest_categorical("num_train_epochs", [4, 5, 6, 10, 20]) # number of epochs tested: 3,4,5,6
    batch_size = 8 # batch size per device during training 
    gradient_accumulation_steps = trial.suggest_categorical("gradient_accumulation_steps", [2, 4, 8]) # number of gradient accumulation steps - tested: 2,4,8
    lora_target = trial.suggest_categorical("lora_target", ["all", '["v_proj", "q_proj"]']) # possible values: "all" - train all possible modules , '["v_proj", "q_proj"]' - only train specific layers

    if lora_target == '["v_proj", "q_proj"]':
        lora_target = ["v_proj", "q_proj"]

    # Copy config and modify parameters (optuna will pick them based on the suggestions)
    config = copy.deepcopy(base_config)
    config.update({
        "learning_rate": lr,
        "lora_rank": rank,
        "lora_alpha": alpha,
        "num_train_epochs": epochs,
        "per_device_train_batch_size": batch_size,
        "output_dir": trial_output_dir,
        "warmup_ratio": warmup_rate,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "lora_target": lora_target
    })

    print(f"Starting trial {trial.number} with lora configs: \n Learning Rate: {lr},\n Lora Rank: {rank},\n Lora Alpha: {alpha},\n Epochs: {epochs},\n Batch Size: {batch_size},\n Warmup Ratio: {warmup_rate},\n Gradient Accumulation Steps: {gradient_accumulation_steps}\n Lora Target: {lora_target}")

    # Save training args into a .json file
    with open(f"{trial_output_dir}/training_args.json", "w") as f:
        json.dump(config, f)
    
    # Save to a temporary config file
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp_config_file:
        json.dump(config, tmp_config_file)
        tmp_config_file_path = tmp_config_file.name

    # Run llamafactory-cli train with the temporary config file
    cmd = ["llamafactory-cli", "train", tmp_config_file_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    tmp_trainer_log = f"{trial_output_dir}/trainer_log.jsonl"

    # Read the trainer log to extract training and evaluation losses
    with open(tmp_trainer_log, "r") as f:
        lines = f.readlines()
        train_losses = []
        eval_losses = []
        train_steps = []
        eval_steps = []
        for line in lines:
            log_data = json.loads(line)
            if "loss" in log_data:
                train_losses.append(log_data["loss"])
                train_steps.append(log_data["current_steps"])
            if "eval_loss" in log_data:
                eval_losses.append(log_data["eval_loss"])
                eval_steps.append(log_data["current_steps"])

    # add ending eval loss and min eval loss to the config for logging purposes
    ending_eval_loss = eval_losses[-1] if eval_losses else None
    min_eval_loss = min(eval_losses) if eval_losses else None
    config["ending_eval_loss"] = ending_eval_loss
    config["min_eval_loss"] = min_eval_loss

    print(f"Trial finished with eval_loss={ending_eval_loss}, min_eval_loss={min_eval_loss}")

    # load the trained model and evaluate on the validation dataset to get the MAP dimensions prediction accuracy

    tokenizer = AutoTokenizer.from_pretrained(trial_output_dir)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left" 

    model = AutoModelForCausalLM.from_pretrained(
            trial_output_dir,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            _attn_implementation="flash_attention_2"
            )

    start_time = time.time()

    generate_text = pipeline(
            model=model,
            #device_map="auto",
            tokenizer=tokenizer,
            task='text-generation',
            return_full_text=True,   # Important for parsing logic
            temperature=0.001,
            do_sample=True,      
            max_new_tokens=64
    )

    model_output = run_batched_pipeline_interference(
            batch=validation_data,
            batch_size=512,
            generate_text_pipeline=generate_text,
            output_column="llm_full_output"
    )

    print(f"Prompting time: {(time.time() - start_time) / 60:.2f} minutes for {len(validation_data)}")

    model_output = validation_data.add_column("llm_full_output", model_output["llm_full_output"])

    #free gpu memory
    del model, tokenizer, generate_text
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
        
    # Step 2: Apply JSON extraction to the LLM's raw output
    model_output = model_output.map(extract_json)
        
    # Step 3: Evaluate the performance
    try:
        # Convert to DataFrame
        model_output = pd.DataFrame(model_output)
        # save the model_output dataframe with predictions into an excel file for error analysis
        model_output.to_excel(f"{trial_output_dir}/validation_results_optuna_run_{trial.number}.xlsx", index=False)
        # Evaluate LLM predictions
        evaluation_result = evaluate_llm(model_output, model_id=f"optuna_run_{trial.number}")

    except Exception as e:
            print(f"Error during evaluation of trial {trial.number}: {e}")
            evaluation_result = None
    # Save evaluation results and return the dimension micro f1 score as the objective to maximize
    if evaluation_result is not None:
        # Save evaluation results to a summary Excel file
        if os.path.exists(f"GLLM/validation_summary_total.xlsx"):
            results_df = pd.read_excel(f"GLLM/validation_summary_total.xlsx")
            results_df = pd.concat([results_df, pd.DataFrame([evaluation_result])], ignore_index=True)
            results_df.to_excel(f"GLLM/validation_summary_total.xlsx", index=False)
        else:
            results_df = pd.DataFrame([evaluation_result])
            results_df.to_excel(f"GLLM/validation_summary_total.xlsx", index=False)
        # Save evaluation results to a summary Excel file (including training arguments) for the current Optuna run
        evaluation_result_2 = {**config, **evaluation_result}
        if os.path.exists(f"{optuna_output_dir}/optuna_run_validation_results.xlsx"):
            results_df = pd.read_excel(f"{optuna_output_dir}/optuna_run_validation_results.xlsx")
            results_df = pd.concat([results_df, pd.DataFrame([evaluation_result_2])], ignore_index=True)
            results_df.to_excel(f"{optuna_output_dir}/optuna_run_validation_results.xlsx", index=False)
        else:
            results_df = pd.DataFrame([evaluation_result_2])
            results_df.to_excel(f"{optuna_output_dir}/optuna_run_validation_results.xlsx", index=False)


        print(f"Fine tuned model with trial number {trial.number} finished with dimension micro f1 score: {evaluation_result['dimension_micro_f1']}")
        return evaluation_result["dimension_micro_f1"]
    else:
        return 0.0

In [ ]:
# Set up the sampler (it uses Tree-structured Parzen Estimator, which picks hyperparameters based on previous results. Alternatively, we can use a RandomSampler)
sampler = optuna.samplers.TPESampler(seed=42)  # the seed assures deterministic for reproducibility

# Alternative we can use a random sampler
#sampler = optuna.samplers.RandomSampler(seed=42)

# create the study 
study = optuna.create_study(direction="maximize", sampler=sampler)

# load the validation dataset
file_name = "GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx"
validation_df = pd.read_excel(file_name)

validation_df = datasets.Dataset.from_pandas(validation_df)

validation_df = validation_df.rename_column("LLM_Confidence_Score", "Confidence_Score")

# start the hyperparameter optimization
study.optimize(lambda trial: objective(trial, validation_data=validation_df), n_trials=30)

Lastly, we will plot the training and validation loss curves for the best (20) and four additional exemplary trials (4, 15, Alternative_1, and Alternative_2). Alternative 1 and Alternative 2 stem from previous tests.

In [ ]:
# Creat Plot folder if it does not exist
if not os.path.exists("Plots"):
    os.makedirs("Plots")

custom_color_palette  = ['#377eb8', '#ff7f00', '#4daf4a','#a65628', '#984ea3', '#e41a1c', '#dede00']

#Next, we will plot the training and validation loss curves for the best (20) and three additional exemplary trials (0, 7, 15) in one figure .
trials_to_plot = [4, 15, 20, "Alternative_1", "Alternative_2"]  # Add the trial numbers you want to plot (including the best one as last entry)

# plot the training and validation loss curves into one plot with different colors for each trial
# make sure that the training and validation loss curves for the same trial have the same color but different line styles (solid for training, dashed for validation) and add a legend to distinguish them
# just plot until 1000 steps for better visibility since some trials have very long training logs
# Use Plotnine for better visualization

# Set up the plotnine theme
plotnine.options.figure_size = (12, 6)

# Create an empty DataFrame to hold all the data for plotting
df_all = pd.DataFrame()

# Loop through the specified trials and extract training and evaluation loss data
for trial_number in trials_to_plot:
    trainer_log_path = f"{optuna_output_dir}/trial_{trial_number}/trainer_log.jsonl"
    if os.path.exists(trainer_log_path):
        with open(trainer_log_path, "r") as f:
            lines = f.readlines()
            train_losses = []
            eval_losses = []
            train_steps = []
            eval_steps = []
            for line in lines:
                log_data = json.loads(line)
                if "loss" in log_data:
                    train_losses.append(log_data["loss"])
                    train_steps.append(log_data["current_steps"])
                if "eval_loss" in log_data:
                    eval_losses.append(log_data["eval_loss"])
                    eval_steps.append(log_data["current_steps"])

        # Use the same color for training and validation curves of the same trial
        color = custom_color_palette[trials_to_plot.index(trial_number) % len(custom_color_palette)]  # Cycle through colors if more than 4 trials

        # Remove points with steps greater than 1000 for better visibility
        train_steps = [step for step in train_steps if step <= 1000]
        train_losses = train_losses[:len(train_steps)]
        eval_steps = [step for step in eval_steps if step <= 1000]
        eval_losses = eval_losses[:len(eval_steps)]
        # Plot training and evaluation loss curves with different line styles using plotnine
        df_plot = pd.DataFrame({
            "Steps": train_steps + eval_steps,
            "Loss": train_losses + eval_losses,
            "Type": ["Train"] * len(train_steps) + ["Validation"] * len(eval_steps),
            "Trial": [f"Trial {trial_number+1}"] * (len(train_steps) + len(eval_steps)) if isinstance(trial_number, int) else [f"{trial_number.replace('_', ' ')}"] * (len(train_steps) + len(eval_steps))
        })

        df_all = pd.concat([df_all, df_plot], ignore_index=True)

# Convert the Trial column to a Categorical type with the specific order
trials_order = df_all['Trial'].unique().tolist() # Get the unique trial names in the order they appear
df_all['Trial'] = pd.Categorical(df_all['Trial'], categories=trials_order, ordered=True)

# Create the plot using plotnine
# make sure to order the legend by trial number and put the legend on the right side of the plot (in the order of the trials_to_plot list)
p = (plotnine.ggplot(df_all, plotnine.aes(x="Steps", y="Loss", color="Trial", linetype="Type"))
     + plotnine.geom_line(size=1.5)
     + plotnine.scale_color_manual(values=custom_color_palette)
     + plotnine.labs(x="Training Steps",  y="Loss")
     + plotnine.theme_minimal()
     + plotnine.theme(
            legend_position="right",
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold'))
     + plotnine.scale_linetype_manual(values={"Train": "solid", "Validation": "dashed"})
     + plotnine.labs(color="Trial", linetype="Loss Type")
)

# Save the plot
if not os.path.exists("Analyses_outputs/Plots/Fine_Tuning"):
    os.makedirs("Analyses_outputs/Plots/Fine_Tuning")

p.save(f"Analyses_outputs/Plots/Fine_Tuning/training_validation_loss_comparison_Local.png", width=19.2, height=9.67, dpi=300)

# Print the plot to the console
p.draw()